In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor, PATH_RESULTS
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 15

eval_bs = 1000

stockGPT, stockGPT_params, opt1, sca1, sch1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3181568
5376
Continuing from previous checkpoint...


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 11:

Learning Rate: 2.00e-04



|██        | 20.0% (02:49) Evaluating model on validation data... (167/168) [2520/12600]:                 

Epoch 11:
Training Loss:
   (MAE) 0.007292722351849079
   (NLL) -3.4081835746765137
Validation Loss:
   (MAE) 0.007393458392471075
   (NLL) -3.4286465644836426

Best Validation: -3.906731128692627
----------------------------------------------------------------------------------------------------

Learning Rate: 2.00e-04



...

|██████    | 60.1% (01:34) Evaluating model on validation data... (167/168) [7560/12600]: 

Epoch 13:
Training Loss:
   (MAE) 0.026224935427308083
   (NLL) -0.11084222793579102
Validation Loss:
   (MAE) 0.0259341262280941
   (NLL) 3.7231717109680176

Best Validation: 3.7231717109680176
----------------------------------------------------------------------------------------------------

Learning Rate: 5.00e-05



|████████  | 80.0% (02:05) Evaluating model on validation data... (167/168) [10080/12600]: 

Epoch 14:
Training Loss:
   (MAE) 0.024844776839017868
   (NLL) -0.461505651473999
Validation Loss:
   (MAE) 0.02468697354197502
   (NLL) 1.8695656061172485

Best Validation: 1.8695656061172485
----------------------------------------------------------------------------------------------------

Learning Rate: 5.00e-05



Epoch 15:
Training Loss:
   (MAE) 0.022782722488045692
   (NLL) -0.7755777835845947
Validation Loss:
   (MAE) 0.022748056799173355
   (NLL) 0.44183647632598877

Best Validation: 0.44183647632598877
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [5]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)


|██████████| 100.0% (01:04) Evaluating model on testing data... (98/99) [3801/3801]:                      

In [6]:
for key, features in [("NLL", StockGPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]),
                      ("MAE", StockGPT_cfg["target_features"]),
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockGPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockGPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-B5: 3.2M
    Training:       -4.1186  -3.9880  -4.0021  -3.9423    >  -4.0128
    Validation:     -4.1254  -4.0284  -4.0361  -4.0019    >  -4.0479
    Testing:        -4.1543  -4.0695  -4.0796  -4.0489    >  -4.0881
    
LinearModel-B5: 5.4K
    Training:       -2.0933  0.3802   -0.0328  -1.3565    >  -0.7756
    Validation:     -2.1366  5.2608   -0.0052  -1.3517    >  0.4418
    Testing:        -2.1747  0.4101   0.0161   -1.3583    >  -0.7767
    
NaiveModel-B5: 0
    Training:       0.9190   0.9190   0.9190   0.9190     >  0.9190
    Validation:     0.9190   0.9190   0.9190   0.9190     >  0.9190
    Testing:        0.9190   0.9190   0.9190   0.9190     >  0.9190
    

------------------------

In [7]:
store_result(PATH_RESULTS, process_result(stockGPT, gpt_losses, gpt_test_losses, max_epochs))
store_result(PATH_RESULTS, process_result(linearModel, linear_losses, linear_test_losses, max_epochs))
store_result(PATH_RESULTS, process_result(naiveModel, naive_losses, naive_test_losses, max_epochs))

print(pd.read_parquet(PATH_RESULTS))

{'model': 'StockGPT-B5', 'bar_width': 5, 'train': {'NLL': [-4.11863899230957, -3.9879794120788574, -4.002130031585693, -3.9423394203186035], 'STD': [0.0058054206892848015, 0.006371809169650078, 0.006301476154476404, 0.006621633656322956], 'MAE': [0.0029863533563911915, 0.0034255459904670715, 0.0034282493870705366, 0.0035948464646935463], 'PMAE': [0.31556040048599243, 0.3506155014038086, 0.3702532649040222, 0.37358900904655457]}, 'val': {'NLL': [-4.125357151031494, -4.028367042541504, -4.036093235015869, -4.001903057098389], 'STD': [0.005969328340142965, 0.006426852196455002, 0.006354149896651506, 0.00660490058362484], 'MAE': [0.003079840447753668, 0.003406079253181815, 0.003396221436560154, 0.003470535157248378], 'PMAE': [0.32691681385040283, 0.35449647903442383, 0.3721939027309418, 0.370632141828537]}, 'test': {'NLL': [-4.154265403747559, -4.0695295333862305, -4.079579830169678, -4.048922061920166], 'STD': [0.005569716915488243, 0.005973591934889555, 0.0058956509456038475, 0.006121114

|██████████| 100.0% (01:23) Evaluating model on testing data... (98/99) [3801/3801]: 